# Single-paper pipeline check

Try the LangTrend detection pipeline on any single paper — not just the ones in a weekly fetch.
Runs the same abstract → HTML → PDF detection cascade `process_papers.py` uses, and (optionally)
the LLM judge stage on top. Two ways to pick a paper:

- **Option A** — an arXiv id (fetches metadata, then abstract → HTML → PDF cascade)
- **Option B** — a PDF already on disk, e.g. a paper not on arXiv (PDF-only detection)

Run from within `notebooks/` (paths are relative, e.g. `../data/...`). Needs the language
taxonomy submodule (`git submodule update --init --recursive` + `python scripts/extract_language_data.py`
if you haven't already) — see the repo root README.

Caches and results are written to `../data/sandbox/<paper-id>/`, separate from the real weekly data.
This is exactly what `scripts/test_single_paper.py` / `make test-paper ARXIV_ID=...` (or
`make test-paper PDF_PATH=...`) do on the command line — this notebook just lets you poke at the
intermediate results interactively.

In [ ]:
import sys, json
from pathlib import Path

sys.path.insert(0, "..")
sys.path.insert(0, "../scripts")

from process_papers import _process_single_paper, load_language_data, _DEFAULT_LANG_DATA
from test_single_paper import (
    fetch_paper_metadata, normalize_arxiv_id, print_report,
    process_local_pdf, slugify_filename,
)

lang_classes, languages_to_ignore, possible_false_positive_languages = load_language_data(_DEFAULT_LANG_DATA)
print(f"Loaded {sum(len(v) for v in lang_classes.values())} language entries across {len(lang_classes)} classes")

## Option A: pick a paper from arXiv

Paste any arXiv id, versioned id, or full `arxiv.org` URL. Skip to **Option B** below instead
if you want to analyze a PDF you already have on disk.

In [ ]:
ARXIV_ID = "1111.11111"  # <- change me

arxiv_id = normalize_arxiv_id(ARXIV_ID)
paper = fetch_paper_metadata(arxiv_id)
safe_id = paper["id"].split("/")[-1]
print(f"{paper['title']}\n{paper['id']}")

### Run the detection cascade (abstract → HTML → PDF)

Set `NO_PDF = True` to skip the (slow, docling-based) PDF fallback while iterating.

In [ ]:
NO_PDF = False

sandbox_dir = Path("../data/sandbox") / safe_id
html_cache_dir = sandbox_dir / "html_cache"
pdf_cache_dir = sandbox_dir / "pdf_cache"
pdf_dir = Path("../data/raw/pdfs")
for d in (html_cache_dir, pdf_cache_dir, pdf_dir):
    d.mkdir(parents=True, exist_ok=True)

if not NO_PDF:
    from langtrend.pdf_processor import init_docling
    init_docling()

record = _process_single_paper(
    paper, lang_classes, languages_to_ignore, possible_false_positive_languages,
    pdf_dir, html_cache_dir, pdf_cache_dir, no_pdf=NO_PDF,
)
print_report(record)

## Option B: analyze a PDF you already have

For a paper that isn't on arXiv (or one you already downloaded) — skips straight to PDF text
extraction and language detection, no abstract or HTML to scan. Run this *instead of* Option A
above, then continue below — both options produce the same `record` / `safe_id` / `sandbox_dir`
variables the rest of the notebook uses.

In [ ]:
PDF_PATH = Path("~/Downloads/some_paper.pdf").expanduser()  # <- change me
TITLE = None  # <- optionally set a title; defaults to the filename

safe_id = slugify_filename(PDF_PATH.stem)
title = TITLE or PDF_PATH.stem

sandbox_dir = Path("../data/sandbox") / safe_id
pdf_cache_dir = sandbox_dir / "pdf_cache"

record = process_local_pdf(
    PDF_PATH, safe_id, title,
    lang_classes, languages_to_ignore, possible_false_positive_languages,
    pdf_cache_dir,
)
print_report(record)

## Inspect a cached section's cleaned text

Useful for seeing exactly what text the language matcher saw, e.g. to debug a surprising
detection or a false positive. Pick a section name printed above.

In [ ]:
SECTION = next(iter(record["sections"]), None)  # <- change me to a specific section name

if SECTION is None:
    print("No sections with detections to inspect.")
elif record["sections"][SECTION]["source"] == "html":
    html_cache = json.loads((html_cache_dir / f"{safe_id}.json").read_text())
    print(html_cache[SECTION]["cleaned_text"][:2000])
else:
    pdf_cache = json.loads((pdf_cache_dir / f"{safe_id}.json").read_text())
    print(pdf_cache["screened_text"][:2000])

## Judge it (optional)

Runs the same LLM-as-judge stage as `judge_check.ipynb`, scoped to this one paper.
Configuration comes from `../.env` — see `.env.example` at the repo root. The default backend
is Cerebras's free tier; point `LLM_JUDGE_BASE_URL` at a local Ollama server for quota-free testing.

In [ ]:
from langtrend.judge import collect_target_languages, assemble_context, build_messages, judge_paper, safe_paper_id
from langtrend.llm_client import LLMClientConfig, OpenAICompatClient

config = LLMClientConfig.from_env()
client = OpenAICompatClient(config)
client.ping()  # raises with an actionable message if the endpoint/key is bad
print(f"Judge: {config.model} @ {config.base_url}")

In [ ]:
targets = collect_target_languages(record)
context = assemble_context(record, sandbox_dir, targets, max_chars=config.max_context_chars)
print(f"Targets: {', '.join(t['language'] for t in targets)}")
print(f"Context: {context.total_chars} chars, coverage={context.coverage}, {len(context.snippets)} snippet(s)")

In [ ]:
judge_record = judge_paper(record, sandbox_dir, client, config)

if judge_record is None:
    print("No languages need judging.")
else:
    width = max(len(name) for name in judge_record["verdicts"]) if judge_record["verdicts"] else 10
    print(f"model: {judge_record['judge_model']}  coverage: {judge_record['context_coverage']}")
    print()
    for target in targets:
        v = judge_record["verdicts"].get(target["language"])
        verdict = v["verdict"] if v else "(unjudged)"
        reason = v["reason"] if v else ""
        print(f"{target['language']:<{width}}  | class {target['class']}  | {verdict:<15} \n{reason}\n--------------------")